# Edge AI Speech-to-Text: Wav2Vec2 Evaluation

In this notebook, we evaluate an alternative ML model, **Wav2Vec2**. It is an acoustic model that is "toned down" compared to Whisper's heavy Transformer-Decoder architecture. It's significantly faster and more lightweight for edge deployment, while still capturing the ML capabilities we want for on-device inference.

In [1]:
import torch
import librosa
import time
from transformers import pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
if torch.backends.mps.is_available():
    device = "mps"
print(f"Executing on device: {device}")

/Users/dimitrimontgomery/text_to_speech_extention/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Executing on device: mps


In [2]:
print("Loading Wav2Vec2 pipeline...")
pipe = pipeline(
    "automatic-speech-recognition",
    model="facebook/wav2vec2-base-960h",
    device=device
)
print("Model loaded successfully!")

Loading Wav2Vec2 pipeline...


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device set to use mps


Model loaded successfully!


In [3]:
audio_path = "test_dictation.wav"
try:
    speech, sr = librosa.load(audio_path, sr=16000)
    
    print("Running warmup (compiling model graphs)...")
    pipe(speech[:16000]) # 1 second warmup
    
    start_time = time.time()
    prediction = pipe(speech)
    end_time = time.time()
    
    raw_text = prediction['text'].strip()
    # Wav2Vec2-base-960h outputs all caps without punctuation.
    # We can apply basic casing logic:
    formatted_text = raw_text.capitalize() + "."
    
    print("\n--- RESULTS ---")
    print(f"Raw Output: '{raw_text}'")
    print(f"Formatted Output: '{formatted_text}'")
    print(f"True Inference Time: {end_time - start_time:.2f} seconds")
except Exception as e:
    print(f"Error processing audio: {e}")

Running warmup (compiling model graphs)...



--- RESULTS ---
Raw Output: 'THE QUICK BROWN FOX JUMPS OVER THE LAZY DOG IN THE MIDDLE OF THE SUMMER'
Formatted Output: 'The quick brown fox jumps over the lazy dog in the middle of the summer.'
True Inference Time: 0.18 seconds
